# Technical Support Assistant with LlamaIndex + LlamaParse
The Gen Academy | Mastering Agentic AI | Approx. 10-minute walkthrough

**Problem:** support engineers must connect a customer's error to the right version of documentation across a large, documented codebase. We demonstrate the pattern on four small technical PDFs for a fictional connector, AtlasSync. These are representative training fixtures, not a real large repository or a benchmark.

**Story:** a version 2.0 connector fails with `E_CONN_104`. Search the docs, inspect sample settings, calculate retry wait, then follow up after one setting changes.

**Why ingest?** A reusable collection needs parsed tables, chunks, embeddings and source/version metadata. Otherwise each ticket requires manually finding and copying the right evidence. This demo includes a conflicting old-version guide to make the version issue visible. The PDF module paths link documentation to conceptual code ownership; we do not index source code or execute a customer application here.

| Component | Purpose | Cost in this demo |
|---|---|---|
| LlamaIndex framework | Data preparation, retrieval and agents | Open-source framework; no hosted credits for these local components |
| LlamaParse via LlamaCloud SDK | PDF to Markdown | Provided LlamaCloud credits, subject to account terms |
| OpenAI | Embeddings and model responses | Separate OpenAI API billing |

LangChain offers general model/tool integrations and configurable agents; LlamaIndex is a useful starting point for data ingestion and retrieval with agents. Both support RAG and agents. LlamaIndex is not the Llama model.

Unzip the complete pack and open this notebook from its folder using a Python 3.12 Jupyter kernel. Keep `docs`, `sample_data` and `manifest.json` alongside it. All data is fictional. No key is stored in the notebook.


## 1 · Credited account and setup (prepare before recording)
- Follow the cohort's actual redemption instructions; select the organization/project with that benefit at https://cloud.llamaindex.ai.
- Check Plan & Credits and Usage in organization settings (labels may vary). Verify the allowance, expiry and overage rules shown in your account; no cohort-specific entitlement is assumed here.
- Create a key for the credited project, off camera. If needed, set `LLAMA_CLOUD_PROJECT_ID` and `LLAMA_CLOUD_BASE_URL` in your environment for your project/region.
- Note starting usage. We submit from this notebook; clicking Run Parse in the browser too would submit a separate job.
- Install before filming; restart the kernel if necessary. `%pip` installs into the current kernel. A ChatGPT subscription is separate from OpenAI API billing.


In [ ]:
from google.colab import files

uploaded = files.upload()

Saving LlamaIndex_Technical_Support_Demo_Pack.zip to LlamaIndex_Technical_Support_Demo_Pack.zip


In [ ]:
from pathlib import Path
import zipfile
import os

zip_path = Path(
    "/content/LlamaIndex_Technical_Support_Demo_Pack.zip"
)

with zipfile.ZipFile(zip_path, "r") as archive:
    archive.extractall("/content")

os.chdir("/content/technical_support_demo")

print("Working folder:", Path.cwd())
print("Files:", [p.name for p in Path.cwd().iterdir()])

Working folder: /content/technical_support_demo
Files: ['manifest.json', 'sample_data', 'LlamaIndex_Technical_Support_Demo.ipynb', 'docs', 'README.txt', 'Demo_Script.txt']


In [ ]:
from pathlib import Path

assert Path("manifest.json").exists()
assert Path("sample_data/connection_profiles.json").exists()

pdfs = list(Path("docs").glob("*.pdf"))
print("PDFs found:", len(pdfs))

for pdf in pdfs:
    print(pdf.name)

PDFs found: 4
02_Troubleshooting_v2.pdf
03_Release_Notes_v2.pdf
04_Archived_Reference_v1.pdf
01_Connector_Reference_v2.pdf


In [ ]:
import os
from google.colab import userdata

os.environ["LLAMA_CLOUD_API_KEY"] = userdata.get(
    "LLAMA_CLOUD_API_KEY"
)

os.environ["OPENAI_API_KEY"] = userdata.get(
    "OPENAI_API_KEY"
)

print("API keys loaded successfully.")

API keys loaded successfully.


In [ ]:
%pip install -q llama-index-core==0.14.25 llama-index-llms-openai==0.8.3 llama-index-embeddings-openai==0.7.0 llama-cloud==2.19.0


## 2 · Important imports
| Import | Role |
|---|---|
| `AsyncLlamaCloud` | Upload/parse PDFs and fetch job usage |
| `Document` | Parsed page plus metadata |
| `IngestionPipeline`, `SentenceSplitter` | Run chunking and embedding transforms |
| `VectorStoreIndex` | Index embedded nodes for retrieval |
| `MetadataFilters`, `MetadataFilter` | Restrict retrieval to the correct version |
| `OpenAIEmbedding`, `OpenAI` | Search vectors versus answer/tool-selection model |
| `FunctionTool` | Expose documentation search and Python functions |
| `FunctionAgent` | Model → tool → result loop |
| `Context`, `Memory`, `ChatMessage` | Workflow state, message memory, role/content representation |

Standard-library imports handle paths, hashes, hidden key entry and JSON. `await` works directly in Jupyter; do not use `asyncio.run()` around these cells.


In [ ]:
import os, json, hashlib, math
from pathlib import Path
from getpass import getpass
from importlib.metadata import version
from llama_cloud import AsyncLlamaCloud
from llama_index.core import Document, VectorStoreIndex
from llama_index.core.ingestion import IngestionPipeline
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core.tools import FunctionTool
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.core.workflow import Context
from llama_index.core.memory import Memory
from llama_index.core.llms import ChatMessage

for name in ("LLAMA_CLOUD_API_KEY", "OPENAI_API_KEY"):
    if not os.environ.get(name):
        os.environ[name] = getpass(f"{name}: ")
llm = OpenAI(model="gpt-4o-mini", temperature=0)
embed_model = OpenAIEmbedding(model="text-embedding-3-small")
for package in ("llama-index-core", "llama-cloud"):
    print(package, version(package))


llama-index-core 0.14.25
llama-cloud 2.19.0


## 3 · Inspect the documentation collection
Open the PDFs before running. There are three 2.0 documents and one archived 1.0 reference. Each has a table plus prose. The manifest supplies trusted source metadata for this small demo; a real system would derive and validate that metadata from repository/release records.

Choose `cost_effective` for this small, mostly textual sample. Verify current rates in the Parse settings/pricing page before submitting. Higher tiers may suit complex layouts, but do not choose one just to spend credits. This demo processes all four files once, including the archive so we can show filtering.


In [ ]:
manifest = json.loads(Path("manifest.json").read_text())
for item in manifest:
    assert Path(item["path"]).exists(), f"Missing {item['path']}; unzip the complete pack."
    print(item["path"], "version", item["product_version"], "module", item["module"])


docs/01_Connector_Reference_v2.pdf version 2.0 module src/connector/client.py
docs/02_Troubleshooting_v2.pdf version 2.0 module src/connector/connection.py
docs/03_Release_Notes_v2.pdf version 2.0 module src/connector/retry.py
docs/04_Archived_Reference_v1.pdf version 1.0 module legacy/connector/client.py


## 4 · LlamaParse: PDFs → Markdown (uses provided credits)
Local output caching avoids resubmitting unchanged documents during rehearsal. The key includes PDF bytes, parsing settings and the supplied project ID. It contains no credentials. A cache hit means **no new parse submission**; its job ID refers to the earlier job. SDK server caching and actual billing can differ from local cache behavior.

If a job times out, inspect the dashboard before retrying: the remote job may still exist. Saved outputs are real service results, never placeholder Markdown. Do not use a hand-written text fallback and then claim to have demonstrated LlamaParse.


In [ ]:
TIER, PARSE_VERSION = "cost_effective", "latest"
project_id = os.environ.get("LLAMA_CLOUD_PROJECT_ID")
scope = {"project_id": project_id} if project_id else {}
client_options = {}
if os.environ.get("LLAMA_CLOUD_BASE_URL"):
    client_options["base_url"] = os.environ["LLAMA_CLOUD_BASE_URL"]
client = AsyncLlamaCloud(**client_options)
cache_dir = Path(".demo_cache")
cache_dir.mkdir(exist_ok=True)
parsed_files = []
for item in manifest:
    path = Path(item["path"])
    settings = json.dumps([TIER, PARSE_VERSION, project_id, client_options], sort_keys=True)
    fingerprint = hashlib.sha256(path.read_bytes() + settings.encode()).hexdigest()
    cache_path = cache_dir / f"parse_{fingerprint}.json"
    if cache_path.exists():
        parsed = json.loads(cache_path.read_text())
        print("REUSED:", path.name)
    else:
        uploaded = await client.files.create(file=path, purpose="parse", **scope)
        result = await client.parsing.parse(
            file_id=uploaded.id, tier=TIER, version=PARSE_VERSION,
            expand=["markdown", "usage"], **scope,
        )
        if result.job.status != "COMPLETED" or not result.markdown or not result.markdown.pages:
            raise RuntimeError(f"No completed Markdown for {path.name}; inspect the job before retrying.")
        parsed = {"job_id": result.job.id,
                  "pages": [p.model_dump(mode="json") for p in result.markdown.pages]}
        cache_path.write_text(json.dumps(parsed, indent=2))
        print("PARSED:", path.name)
    parsed_files.append({"source": item, "result": parsed})
print(parsed_files[0]["result"]["pages"][0]["markdown"])


PARSED: 01_Connector_Reference_v2.pdf
PARSED: 02_Troubleshooting_v2.pdf
PARSED: 03_Release_Notes_v2.pdf
PARSED: 04_Archived_Reference_v1.pdf
# **AtlasSync | Connector Reference**

Fictional technical documentation | Product version 2.0 | Demo only

Module: src/connector/client.py

## **What this component does**

AtlasSync is a fictional service that copies application records into a reporting system. The connector opens a connection to the AtlasSync server. This document covers AtlasSync 2.0 only.

## **Connection settings**

<table>
  <tr>
    <th>Setting</th>
    <th>Required value for 2.0</th>
  </tr>
<tr>
    <td>Endpoint port</td>
    <td>8443</td>
  </tr>
<tr>
    <td>TLS encryption</td>
    <td>Enabled (true)</td>
  </tr>
<tr>
    <td>Connection timeout</td>
    <td>10 seconds per attempt</td>
  </tr>
</table>

## **Connection requirements**

For AtlasSync 2.0, use port 8443 and enable TLS. TLS encrypts the connection. Port 8080 is the old 1.0 endpoint and is not a supported 2.

## 5 · Verify the output and actual credit usage
Compare PDF and Markdown: version 2.0 requires port **8443** and TLS **enabled**; version 1.0 used port 8080. Check the retry table too. Do not index incorrect output.

This fetches existing jobs, not new parses. Job credit values may remain null briefly after completion; recheck later rather than parsing again. Compare with organization Usage in the dashboard. Cached jobs describe earlier consumption, not a new deduction. OpenAI costs are separate.


In [ ]:
for entry in parsed_files:
    job = await client.parsing.get(entry["result"]["job_id"], expand=["usage"], **scope)
    credits = job.job.usage.credits if job.job.usage else None
    print(entry["source"]["path"], "job", job.job.id,
          "credits:", credits if credits is not None else "pending")


docs/01_Connector_Reference_v2.pdf job pjb-hsrv6rvfpnt694suynsj7deglktu credits: 3.0
docs/02_Troubleshooting_v2.pdf job pjb-jfbx0y8ehth8fk3gfze9dkkq836y credits: 3.0
docs/03_Release_Notes_v2.pdf job pjb-6lvw170crbp73bfudbesc43mssos credits: 3.0
docs/04_Archived_Reference_v1.pdf job pjb-foae1l647urntiwadvqrftrbt3tq credits: 3.0


## 6 · Actual ingestion pipeline: pages → chunks → embeddings
Keep filename, product version, page and module metadata attached to every chunk. Embeddings support meaning-based search; metadata supports precise restrictions. Chunk size is a token budget, not an embedding dimension.

`IngestionPipeline` runs the transforms explicitly. Nodes already contain embeddings when passed to `VectorStoreIndex`, so the index can reuse them. `num_workers=1` keeps the notebook simple. The in-memory vector store needs no additional account.


In [ ]:
documents = []
for entry in parsed_files:
    item = entry["source"]
    for i, page in enumerate(entry["result"]["pages"]):
        documents.append(Document(
            text=page["markdown"],
            metadata={"file_name": Path(item["path"]).name,
                      "page": page.get("page_number") or i + 1,
                      "product": item["product"], "product_version": item["product_version"],
                      "module": item["module"]},
        ))
pipeline = IngestionPipeline(transformations=[
    SentenceSplitter(chunk_size=384, chunk_overlap=40), embed_model,
])
nodes = await pipeline.arun(documents=documents, num_workers=1)
index = VectorStoreIndex(nodes, embed_model=embed_model)
print("Documents:", len(documents), "Chunks:", len(nodes))
print(nodes[0].metadata)


Documents: 4 Chunks: 4
{'file_name': '01_Connector_Reference_v2.pdf', 'page': 1, 'product': 'AtlasSync', 'product_version': '2.0', 'module': 'src/connector/client.py'}


## 7 · Basic RAG with version filtering and visible evidence
For this demo the customer is confirmed to run AtlasSync 2.0. Filter before retrieval; a prompt alone is not the version boundary. `similarity_top_k=3` retrieves up to three chunks, not three answers. In a larger app, validate the customer's version and permissions before selecting a scoped retriever.


In [ ]:
TARGET_VERSION = "2.0"
filters = MetadataFilters(filters=[
    MetadataFilter(key="product", value="AtlasSync"),
    MetadataFilter(key="product_version", value=TARGET_VERSION),
])
query_engine = index.as_query_engine(llm=llm, similarity_top_k=3, filters=filters)
response = await query_engine.aquery(
    "For AtlasSync 2.0, what port and TLS setting are required, and what does E_CONN_104 mean?"
)
print(str(response))
for source in response.source_nodes:
    print("\nSOURCE:", source.node.metadata)
    print(source.node.get_content())
assert all(s.node.metadata["product_version"] == TARGET_VERSION for s in response.source_nodes)


For AtlasSync 2.0, the required port is 8443, and TLS encryption must be enabled (true). The error code E_CONN_104 indicates that the connector could not establish a connection, which may be due to various issues such as endpoint mismatch, TLS mismatch, network routing problems, or server health concerns.

SOURCE: {'file_name': '01_Connector_Reference_v2.pdf', 'page': 1, 'product': 'AtlasSync', 'product_version': '2.0', 'module': 'src/connector/client.py'}
# **AtlasSync | Connector Reference**

Fictional technical documentation | Product version 2.0 | Demo only

Module: src/connector/client.py

## **What this component does**

AtlasSync is a fictional service that copies application records into a reporting system. The connector opens a connection to the AtlasSync server. This document covers AtlasSync 2.0 only.

## **Connection settings**

<table>
  <tr>
    <th>Setting</th>
    <th>Required value for 2.0</th>
  </tr>
<tr>
    <td>Endpoint port</td>
    <td>8443</td>
  </tr>
<tr>
    

## 8 · Diagnostic and calculator tools
The profiles are static sanitized JSON fixtures; no network probe, repository scan or real configuration change occurs. The diagnostic compares observed settings against expected values the agent must retrieve from the docs. Passing arbitrary expected values would make a comparison unreliable: inspect the retrieved evidence as well as the tool result.

The calculator estimates the fixed-delay retry window from the documented values. Its docstring/type hints expose the tool interface. Tool prints show execution, not private model reasoning.


In [ ]:
profiles = json.loads(Path("sample_data/connection_profiles.json").read_text())

def inspect_connection(profile: str, expected_port: int, required_tls: bool) -> dict:
    """Compare a named sample configuration with requirements retrieved from AtlasSync 2.0 docs. Does not test connectivity."""
    if profile not in profiles:
        raise ValueError("Unknown profile; use reported, port_fixed or settings_fixed.")
    observed = profiles[profile]
    findings = []
    if observed["port"] != expected_port:
        findings.append(f"Port mismatch: observed {observed['port']}, documented {expected_port}.")
    if observed["tls_enabled"] != required_tls:
        findings.append(f"TLS mismatch: observed {observed['tls_enabled']}, documented {required_tls}.")
    result = {"observed": observed, "findings": findings,
              "limit": "Static comparison only; matching settings do not prove a working connection."}
    print("[DIAGNOSTIC]", profile, json.dumps(result))
    return result

def calculate_retry_window(attempts: int, timeout_seconds: float, delay_seconds: float) -> float:
    """Calculate worst-case fixed-delay retry seconds. Attempts includes the first try; there are attempts minus one gaps. Excludes overhead."""
    if attempts < 1 or not all(math.isfinite(x) and x >= 0 for x in [timeout_seconds, delay_seconds]):
        raise ValueError("Use positive attempts and finite nonnegative times.")
    total = attempts * timeout_seconds + (attempts - 1) * delay_seconds
    print(f"[CALCULATOR] {attempts} * {timeout_seconds} + {attempts - 1} * {delay_seconds} = {total} seconds")
    return total

async def search_v2_docs(question: str) -> str:
    """Search version-filtered AtlasSync 2.0 documentation; return answer plus source/page evidence."""
    result = await query_engine.aquery(question)
    evidence = [{"source": s.node.metadata, "text": s.node.get_content()}
                for s in result.source_nodes]
    print("[SEARCH]", [e["source"]["file_name"] for e in evidence])
    return json.dumps({"answer": str(result), "evidence": evidence})

# Explicit wrapper keeps source metadata visible to the agent, not just the answer.
support_search = FunctionTool.from_defaults(async_fn=search_v2_docs)
diagnostic_tool = FunctionTool.from_defaults(fn=inspect_connection)
calculator_tool = FunctionTool.from_defaults(fn=calculate_retry_window)


## 9 · Agent: documentation search + sample diagnostic + retry calculation
RAG follows a retrieve-and-answer path. An agent can choose tools and use their results across steps. This is one agent with three tools, not a multi-agent system. It is deliberately scoped to version 2.0.


In [ ]:
agent = FunctionAgent(
    llm=llm, tools=[support_search, diagnostic_tool, calculator_tool],
    system_prompt=(
        "You are an AtlasSync 2.0 technical support assistant. Keep replies under 150 words. "
        "Search search_v2_docs for technical requirements, error meaning, retry defaults and next steps. "
        "Use the retrieved port/TLS requirements as expected values for inspect_connection. "
        "For retry estimates, obtain the values from docs and call calculate_retry_window. "
        "Use source filename/page references when available in the search result. "
        "State that E_CONN_104 can have multiple causes. Configuration mismatches are findings, not proof of root cause. "
        "Use conversation history for product/version and the active case. "
        "If the user gives a new sample profile, inspect that profile again. "
        "Never claim a live test, successful connection or a real setting change. "
        "If matching settings still fail, follow the runbook's escalation steps. "
        "If a question is not covered, acknowledge the gap and refer to the platform team."
    ),
)
ctx = Context(agent)
memory = Memory.from_defaults(token_limit=12000)
answer = await agent.run(
    "Investigate E_CONN_104 for AtlasSync 2.0. "
    "Complete all three steps before answering: "
    "1. Call search_v2_docs to retrieve the required port, TLS setting, "
    "and default retry parameters. "
    "2. Call inspect_connection with profile='reported', using the "
    "documented port and TLS requirements. Report the actual mismatches. "
    "3. Call calculate_retry_window using the documented retry parameters, "
    "even if the documentation already states the total. "
    "Then summarize the findings and recommended next steps with sources.",
    ctx=ctx,
    memory=memory,
)

print(str(answer))


[SEARCH] ['03_Release_Notes_v2.pdf', '02_Troubleshooting_v2.pdf', '01_Connector_Reference_v2.pdf']
[DIAGNOSTIC] reported {"observed": {"product": "AtlasSync", "product_version": "2.0", "port": 8080, "tls_enabled": false, "error": "E_CONN_104"}, "findings": ["Port mismatch: observed 8080, documented 8443.", "TLS mismatch: observed False, documented True."], "limit": "Static comparison only; matching settings do not prove a working connection."}
[CALCULATOR] 3 * 10 + 2 * 2 = 34 seconds
### Findings on E_CONN_104

1. **Error Meaning**: E_CONN_104 indicates that the connector could not establish a connection. It can arise from various issues, including endpoint or TLS mismatches, network routing problems, or server health concerns (source: 02_Troubleshooting_v2.pdf).

2. **Connection Inspection**:
   - **Port Mismatch**: Observed port is **8080**, but the documented requirement is **8443**.
   - **TLS Mismatch**: TLS is **disabled** (False), while it should be **enabled** (True) (source: 0

**Expected checks, not pre-generated outputs:** port 8080 vs required 8443; TLS false vs required true. Retry window = 3 × 10 + 2 × 2 = **34 seconds**. Both mismatches are observations; they do not prove the unique root cause. Verify the `[DIAGNOSTIC]` and `[CALCULATOR]` prints before narrating tool execution. Tool ordering and wording can vary.

## 10 · Follow-up and conversation memory
The next message omits the error and version. Reuse Context and Memory; inspect the updated sample rather than assuming the previous result still applies.


In [ ]:
follow_up = await agent.run(
    "The customer changed only the port. Use the port_fixed sample profile. What still needs attention?",
    ctx=ctx, memory=memory,
)
print(str(follow_up))


/usr/local/lib/python3.13/dist-packages/workflows/context/state_store.py:231: UnserializableKeyWarning: Skipping serialization of known unserializable key: memory -- This is expected but will require this item to be set manually after deserialization.
  warnings.warn(


[DIAGNOSTIC] port_fixed {"observed": {"product": "AtlasSync", "product_version": "2.0", "port": 8443, "tls_enabled": false, "error": "E_CONN_104"}, "findings": ["TLS mismatch: observed False, documented True."], "limit": "Static comparison only; matching settings do not prove a working connection."}
The inspection of the **port_fixed** sample profile shows that the port is correctly set to **8443**, but there is a **TLS mismatch**: TLS is currently **disabled** (observed False), while it must be **enabled** (documented True) for AtlasSync 2.0.

To resolve the **E_CONN_104** error, ensure that TLS is enabled in the configuration. 

Remember, matching settings alone do not guarantee a successful connection; further testing in a controlled environment is necessary after making these adjustments. If issues persist, collect logs and consider escalating the case.


Expected: port now matches; TLS remains disabled and must be enabled according to v2 docs. No real customer change was performed: we selected another static fixture.

## 11 · Inspect message history; then start a fresh session
Chat/message history contains earlier user, assistant and tool interactions. `ChatMessage` is a role/content representation. Memory manages messages; workflow Context holds runtime state; retrieved context is document evidence. None is permanent model training. No durable cross-restart session storage is configured here.


In [ ]:
history = await memory.aget_all()
for message in history:
    print(f"{message.role.value}: {message.content}")

fresh_ctx = Context(agent)
fresh_memory = Memory.from_defaults(token_limit=12000)
fresh = await agent.run(
    "Which sample profile did I ask you to inspect in this conversation? If I haven't specified one, ask me.",
    ctx=fresh_ctx, memory=fresh_memory,
)
print(str(fresh))


user: Our AtlasSync 2.0 connector reports E_CONN_104. Search the docs and inspect the reported sample profile. Explain the mismatches and next steps. Also calculate the worst-case retry wait using documented defaults.
assistant: 
tool: {"answer": "E_CONN_104 indicates that the connector could not establish a connection. This error does not specify a unique cause, so potential issues may include endpoint mismatch, TLS mismatch, network routing problems, or server health concerns. Each of these areas requires investigation to identify the root cause of the connection failure.", "evidence": [{"source": {"file_name": "02_Troubleshooting_v2.pdf", "page": 1, "product": "AtlasSync", "product_version": "2.0", "module": "src/connector/connection.py"}, "text": "# **AtlasSync | Troubleshooting Runbook**\n\nFictional technical documentation | Product version 2.0 | Demo only\n\nModule: src/connector/connection.py\n\n## **Incident**\n\nE_CONN_104 means the connector could not establish a connection.